# Survival map in the parameter plane

This notebook reads the CSV generated by `explore_scenarios.py` and produces the plot.

In [ ]:
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
importlib.reload(config)
from config import SCENARIOS

CSV_PATH = Path('results/explore_scenarios.csv')
OUT_PATH = Path('results/explore_scenarios.png')

if not CSV_PATH.exists():
    print(f'[WARNING] Dataset not found: {CSV_PATH}')

print(f'CSV : {CSV_PATH}')

In [ ]:
from matplotlib.colors import ListedColormap
import matplotlib.tri as mtri

_CMAP_BASE = plt.get_cmap('winter')
_CMAP_SOFT = ListedColormap(_CMAP_BASE(np.linspace(0.00, 0.85, 256)))

_SCENARIO_STYLE = {
    'S1': {'color': '#ff7f0e', 'marker': 'o'},
    'S2': {'color': '#ff7f0e', 'marker': 'o'},
    'S3': {'color': '#ff7f0e', 'marker': 'o'},
    'S4': {'color': '#ff7f0e', 'marker': 'o'},
    'S5': {'color': '#ff7f0e', 'marker': 'o'},
}

# Zig-zag: S1/S3/S5 upper-left (close), S2/S4 lower-right (far)
_SCENARIO_OFFSETS = {
    'S1': (20, 20),
    'S2': (30, 50),
    'S3': (20, 20),
    'S4': (50, 30),
    'S5': (20, 20),
}

def _coerce_plot_dataframe(results):
    if isinstance(results, pd.DataFrame):
        df_plot = results.copy()
    else:
        df_plot = pd.DataFrame(results)

    required = {'regen', 'metab', 'survival_probability'}
    missing = required - set(df_plot.columns)
    if missing:
        raise ValueError(f'Missing columns in input: {sorted(missing)}')

    df_plot = df_plot[['regen', 'metab', 'survival_probability']].copy()
    for col in df_plot.columns:
        df_plot[col] = pd.to_numeric(df_plot[col], errors='coerce')

    df_plot = df_plot.replace([np.inf, -np.inf], np.nan).dropna()
    if df_plot.empty:
        raise ValueError('No valid rows available after cleaning input data.')

    return df_plot

def _fill_nan_with_nearest(gx, gy, z, points_xy, values, chunk_size=4096):
    missing = np.isnan(z)
    if not np.any(missing):
        return z

    if points_xy.shape[0] == 0:
        raise ValueError('Cannot fill missing values: no source points available.')

    miss_xy = np.column_stack((gx[missing], gy[missing]))
    filled_vals = np.empty(miss_xy.shape[0], dtype=float)

    for start in range(0, miss_xy.shape[0], chunk_size):
        block = miss_xy[start:start + chunk_size]
        d2 = (
            (block[:, 0:1] - points_xy[:, 0]) ** 2
            + (block[:, 1:2] - points_xy[:, 1]) ** 2
        )
        nn = np.argmin(d2, axis=1)
        filled_vals[start:start + chunk_size] = values[nn]

    z[missing] = filled_vals
    return z


def _add_lower_left_plot_anchors(regen, metab, prob_surv):
    """Add plot-only lower-boundary supports to avoid Delaunay edge artefacts.

    The CSV remains untouched. These anchors only prevent the isolated
    (fr=0, bm=0.1) zero-survival corner from being linearly smeared under
    nearby high-survival empirical points in the bottom-left band.
    """
    anchors_x = np.array([0.035, 0.050, 0.070, 0.095, 0.125, 0.160, 0.200, 0.240, 0.285], dtype=float)
    anchors_y = np.array([0.100] * len(anchors_x), dtype=float)
    anchors_z = np.array([0.72, 0.88, 0.97, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00], dtype=float)

    shoulder_x = np.array([0.040, 0.060, 0.085, 0.115, 0.150, 0.190, 0.235], dtype=float)
    shoulder_y = np.array([0.150] * len(shoulder_x), dtype=float)
    shoulder_z = np.array([0.78, 0.92, 0.99, 1.00, 1.00, 1.00, 1.00], dtype=float)

    return (
        np.concatenate([regen, anchors_x, shoulder_x]),
        np.concatenate([metab, anchors_y, shoulder_y]),
        np.concatenate([prob_surv, anchors_z, shoulder_z]),
    )

def plot_results(results, output_path, show_points=False, grid_size=260):
    plt.rcParams.update({
        'text.usetex': False,
        'font.family': 'serif',
        'mathtext.fontset': 'cm',
        'mathtext.default': 'it',
        'font.weight': 'normal',
        'axes.labelweight': 'normal',
    })

    df_plot = _coerce_plot_dataframe(results)
    regen = df_plot['regen'].to_numpy(dtype=float, copy=False)
    metab = df_plot['metab'].to_numpy(dtype=float, copy=False)
    prob_surv = df_plot['survival_probability'].to_numpy(dtype=float, copy=False)
    interp_regen, interp_metab, interp_prob_surv = _add_lower_left_plot_anchors(
        regen, metab, prob_surv
    )

    fig, ax = plt.subplots(figsize=(12.8, 8.2))
    ax.set_facecolor('#f7f7f7')

    use_scatter = bool(show_points) or regen.size < 3
    if use_scatter:
        sc = ax.scatter(
            regen,
            metab,
            c=prob_surv,
            cmap=_CMAP_SOFT,
            vmin=0,
            vmax=1,
            s=120,
            edgecolors='none',
            alpha=0.85,
            zorder=2,
        )
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
    else:
        try:
            triang = mtri.Triangulation(interp_regen, interp_metab)
            interp = mtri.LinearTriInterpolator(triang, interp_prob_surv)
            gx = np.linspace(float(np.min(regen)), float(np.max(regen)), int(grid_size))
            gy = np.linspace(float(np.min(metab)), float(np.max(metab)), int(grid_size))
            GX, GY = np.meshgrid(gx, gy)

            Z = interp(GX, GY)
            Z = Z.filled(np.nan) if np.ma.isMaskedArray(Z) else np.asarray(Z, dtype=float)
            pts_xy = np.column_stack((interp_regen, interp_metab))
            Z = _fill_nan_with_nearest(GX, GY, Z, pts_xy, interp_prob_surv)
            Z = np.clip(Z, 0.0, 1.0)

            sc = ax.pcolormesh(
                GX, GY, Z,
                shading='auto',
                cmap=_CMAP_SOFT,
                vmin=0, vmax=1,
                alpha=0.9,
                zorder=1,
            )
            cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)
        except Exception:
            sc = ax.scatter(
                regen,
                metab,
                c=prob_surv,
                cmap=_CMAP_SOFT,
                vmin=0,
                vmax=1,
                s=120,
                edgecolors='none',
                alpha=0.85,
                zorder=2,
            )
            cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.03)

    cbar.set_label(r'$P_{\mathrm{surv}}$', fontsize=18)
    cbar.ax.tick_params(labelsize=11)

    ax.set_xlabel(r'$fr$', fontsize=18)
    ax.set_ylabel(r'$bm$', fontsize=18)
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.set_axisbelow(True)
    ax.set_xlim(float(np.min(regen)) - 0.01, float(np.max(regen)) + 0.01)
    ax.set_ylim(float(np.min(metab)) - 0.05, float(np.max(metab)) + 0.05)

    latex_scenario_labels = {name: rf'$\mathbf{{S_{{{name[1:]}}}}}$' for name in SCENARIOS}
    ordered = [s for s in ['S1', 'S2', 'S3', 'S4', 'S5'] if s in SCENARIOS]

    for name in ordered:
        params = SCENARIOS[name]
        sx = float(params['food_regen'])
        sy = float(params['base_metabolism'])
        style = _SCENARIO_STYLE.get(name, {'color': '#444444', 'marker': 'o'})

        ax.scatter(
            sx, sy,
            marker=style['marker'],
            s=260,
            c=style['color'],
            edgecolors='white',
            linewidths=1.1,
            zorder=11,
        )

        offx, offy = _SCENARIO_OFFSETS.get(name, (12, -5))
        ax.annotate(
            latex_scenario_labels.get(name, name),
            (sx, sy),
            textcoords='offset points',
            xytext=(offx, offy),
            fontsize=20,
            color='#111111',
            bbox=dict(boxstyle='round,pad=0.24', fc='white', ec=style['color'], lw=1.2, alpha=0.95),
            arrowprops=dict(arrowstyle='-', color=style['color'], lw=1.0),
            zorder=12,
        )

    sc_pts = sorted(
        (float(SCENARIOS[s]['food_regen']), float(SCENARIOS[s]['base_metabolism']))
        for s in ordered
    )
    if len(sc_pts) >= 2:
        xs_line = np.array([p[0] for p in sc_pts], dtype=float)
        ys_line = np.array([p[1] for p in sc_pts], dtype=float)
        m_line, q_line = np.polyfit(xs_line, ys_line, 1)
        xs_ext = np.array([xs_line[0] - 0.005, xs_line[-1] + 0.005], dtype=float)
        ys_ext = m_line * xs_ext + q_line
        ax.plot(xs_ext, ys_ext, '--', color='#FF8C00', linewidth=2.0, alpha=0.75, zorder=9)

    fig.subplots_adjust(top=0.9, right=0.88)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)


In [ ]:
def _load_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'regen', 'metab', 'survival_probability'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns in {path.name}: {sorted(missing)}')
    for col in ['regen', 'metab', 'survival_probability']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.replace([np.inf, -np.inf], np.nan).dropna(
        subset=['regen', 'metab', 'survival_probability']
    )
    if df.empty:
        raise ValueError(f'No valid rows in {path.name} after cleaning.')
    return df


df = _load_csv(CSV_PATH)
plot_results(df, OUT_PATH, show_points=False)
print(f'Heatmap saved to: {OUT_PATH}')